# 04 — Modeling and Ablation

This notebook compares transaction-only and graph-assisted fraud detection models.

Model development is performed using a temporally ordered training and validation
period. The final holdout test period is reserved for the final evaluation notebook.

The experiments also examine whether strong sender-balance features represent
transferable fraud behavior or PaySim-specific shortcuts.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

DATA_DIR = Path("../data")

In [2]:
df = pd.read_csv(
    DATA_DIR / "transaction_features_temporal_v1.csv"
)

print("Shape:", df.shape)
print("Missing:", df.isna().sum().sum())
print("Duplicate tx_id:", df["tx_id"].duplicated().sum())

df.head()

Shape: (200000, 19)
Missing: 0
Duplicate tx_id: 0


,tx_id,src_id,dst_id,time_step,tx_type,amount,log_amount,src_balance_before,dst_balance_before,amount_to_src_balance_ratio,amount_exceeds_src_balance,src_balance_zero,receiver_prior_tx_count,receiver_prior_total_amount,receiver_prior_avg_amount,label_fraud,receiver_prior_24h_tx_count,receiver_prior_7d_tx_count,receiver_prior_7d_total_amount
0,tx_1,C1446786192,C629162506,207,CASH_OUT,201544.43,12.213770,0.0,4427398.04,0.0,1,1,0,0.00,0.00,0,0,0,0.00
1,tx_2,C1507137317,C95898629,301,CASH_OUT,99796.32,11.510897,0.0,2244500.63,0.0,1,1,0,0.00,0.00,0,0,0,0.00
2,tx_3,C689120377,C1777573992,429,CASH_OUT,31901.99,10.370455,0.0,9080459.95,0.0,1,1,0,0.00,0.00,0,0,0,0.00
3,tx_4,C1224308913,C766410750,187,CASH_OUT,286477.31,12.565418,0.0,949477.95,0.0,1,1,1,19989.23,19989.23,0,0,1,19989.23
4,tx_5,C2098350559,C85657936,159,CASH_OUT,119075.98,11.687525,0.0,187137.64,0.0,1,1,0,0.00,0.00,0,0,0,0.00


## 2. Temporal Development Split

Transactions are split chronologically rather than randomly.

The latest 20% of the timeline is reserved as a final holdout test period.
The earlier training period is split again into train-core and validation sets
for model development.

In [3]:
test_cutoff = df["time_step"].quantile(0.80)

train_period = df[
    df["time_step"] <= test_cutoff
].copy()

test_df = df[
    df["time_step"] > test_cutoff
].copy()

print("Final test cutoff:", test_cutoff)
print("Training period:", train_period.shape)
print("Reserved test:", test_df.shape)

Final test cutoff: 356.0
Training period: (160264, 19)
Reserved test: (39736, 19)


In [4]:
validation_cutoff = train_period["time_step"].quantile(0.80)

train_core_df = train_period[
    train_period["time_step"] <= validation_cutoff
].copy()

val_df = train_period[
    train_period["time_step"] > validation_cutoff
].copy()

print("Validation cutoff:", validation_cutoff)
print("Train core:", train_core_df.shape)
print("Validation:", val_df.shape)

Validation cutoff: 302.0
Train core: (128659, 19)
Validation: (31605, 19)


In [5]:
print("TRAIN CORE")
print(train_core_df["label_fraud"].value_counts())

print("\nVALIDATION")
print(val_df["label_fraud"].value_counts())

print("\nRESERVED TEST")
print(test_df["label_fraud"].value_counts())

TRAIN CORE
label_fraud
0    125246
1      3413
Name: count, dtype: int64

VALIDATION
label_fraud
0    31049
1      556
Name: count, dtype: int64

RESERVED TEST
label_fraud
0    35492
1     4244
Name: count, dtype: int64


## 3. Experimental Feature Sets

Four feature groups are compared.

The initial feature sets include sender-balance-derived variables. Later
experiments remove this feature family to test whether the model depends on
PaySim-specific synthetic shortcuts.

In [6]:
dirty_baseline_features = [
    "tx_type",
    "amount",
    "log_amount",
    "src_balance_before",
    "dst_balance_before",
    "amount_to_src_balance_ratio",
    "amount_exceeds_src_balance",
    "src_balance_zero"
]

dirty_graph_features = dirty_baseline_features + [
    "receiver_prior_tx_count",
    "receiver_prior_total_amount",
    "receiver_prior_avg_amount"
]

clean_baseline_features = [
    "tx_type",
    "amount",
    "log_amount",
    "dst_balance_before"
]

clean_graph_features = clean_baseline_features + [
    "receiver_prior_tx_count",
    "receiver_prior_total_amount",
    "receiver_prior_avg_amount"
]

temporal_graph_features = clean_graph_features + [
    "receiver_prior_7d_tx_count",
    "receiver_prior_7d_total_amount"
]

The `clean` feature sets remove the sender-balance feature family:

- `src_balance_before`
- `amount_to_src_balance_ratio`
- `amount_exceeds_src_balance`
- `src_balance_zero`

Post-transaction balances are already excluded from all primary model variants.

In [7]:
def build_logistic_model(feature_columns):
    categorical = ["tx_type"]

    numeric = [
        col for col in feature_columns
        if col not in categorical
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                categorical
            ),
            (
                "num",
                StandardScaler(),
                numeric
            )
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ])

In [8]:
def build_random_forest_model():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["tx_type"]
            )
        ],
        remainder="passthrough"
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

In [9]:
model_results = []

def evaluate_model(
    model_name,
    model,
    feature_columns
):
    X_train = train_core_df[
        feature_columns
    ].copy()

    y_train = train_core_df[
        "label_fraud"
    ].copy()

    X_val = val_df[
        feature_columns
    ].copy()

    y_val = val_df[
        "label_fraud"
    ].copy()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    cm = confusion_matrix(
        y_val,
        y_pred
    )

    model_results.append({
        "model": model_name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    })

    print(model_name)
    print("\nConfusion matrix:")
    print(cm)

    print("\nFraud precision:", round(precision, 4))
    print("Fraud recall:", round(recall, 4))
    print("Fraud F1:", round(f1, 4))
    print("Accuracy:", round(accuracy, 4))

    return model

## 4. Initial Baseline

The first Logistic Regression model includes sender-balance-derived features.

This experiment is retained to demonstrate how apparently strong performance
can depend on dataset-specific variables.

In [10]:
dirty_baseline_model = build_logistic_model(
    dirty_baseline_features
)

dirty_baseline_model = evaluate_model(
    "Dirty Baseline Logistic",
    dirty_baseline_model,
    dirty_baseline_features
)

Dirty Baseline Logistic

Confusion matrix:
[[28277  2772]
 [    2   554]]

Fraud precision: 0.1666
Fraud recall: 0.9964
Fraud F1: 0.2854
Accuracy: 0.9122


In [11]:
dirty_graph_model = build_logistic_model(
    dirty_graph_features
)

dirty_graph_model = evaluate_model(
    "Dirty + Graph Logistic",
    dirty_graph_model,
    dirty_graph_features
)

Dirty + Graph Logistic

Confusion matrix:
[[28285  2764]
 [    2   554]]

Fraud precision: 0.167
Fraud recall: 0.9964
Fraud F1: 0.286
Accuracy: 0.9125


## 5. Sender-Balance Feature Ablation

The sender-balance feature family is removed to reduce dependence on
PaySim-specific synthetic behavior.

The resulting clean models provide a more conservative estimate of
transferable fraud-detection performance.

In [14]:
clean_graph_model = build_logistic_model(
    clean_graph_features
)

clean_graph_model = evaluate_model(
    "Clean + Graph Logistic",
    clean_graph_model,
    clean_graph_features
)

Clean + Graph Logistic

Confusion matrix:
[[26315  4734]
 [  183   373]]

Fraud precision: 0.073
Fraud recall: 0.6709
Fraud F1: 0.1317
Accuracy: 0.8444


In [15]:
temporal_logistic_model = build_logistic_model(
    temporal_graph_features
)

temporal_logistic_model = evaluate_model(
    "Clean + Temporal Graph Logistic",
    temporal_logistic_model,
    temporal_graph_features
)

Clean + Temporal Graph Logistic

Confusion matrix:
[[26397  4652]
 [  185   371]]

Fraud precision: 0.0739
Fraud recall: 0.6673
Fraud F1: 0.133
Accuracy: 0.847


## 6. Nonlinear Model

Random Forest is evaluated on the clean temporal graph feature set.

Unlike Logistic Regression, the model can capture nonlinear interactions
between transaction amount, receiver history, and temporal graph activity.

In [16]:
rf_model = build_random_forest_model()

rf_model = evaluate_model(
    "Random Forest + Temporal Graph",
    rf_model,
    temporal_graph_features
)

Random Forest + Temporal Graph

Confusion matrix:
[[30689   360]
 [  318   238]]

Fraud precision: 0.398
Fraud recall: 0.4281
Fraud F1: 0.4125
Accuracy: 0.9785


In [17]:
comparison_df = (
    pd.DataFrame(model_results)
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)

comparison_df.round(4)

,model,precision,recall,f1,accuracy
0,Random Forest + Temporal Graph,0.3980,0.4281,0.4125,0.9785
1,Dirty + Graph Logistic,0.1670,0.9964,0.2860,0.9125
2,Dirty Baseline Logistic,0.1666,0.9964,0.2854,0.9122
3,Clean + Temporal Graph Logistic,0.0739,0.6673,0.1330,0.8470
4,Clean + Graph Logistic,0.0730,0.6709,0.1317,0.8444


In [18]:
comparison_df.to_csv(
    DATA_DIR / "model_validation_comparison.csv",
    index=False
)

print(
    "Saved: ../data/model_validation_comparison.csv"
)

Saved: ../data/model_validation_comparison.csv


## 7. Modeling Conclusions

The ablation experiments distinguish raw predictive performance from more
transferable fraud signals.

Sender-balance-derived features produce unusually strong performance in PaySim,
but they are treated cautiously because they may reflect synthetic generation
rules.

Receiver-history graph features provide a smaller but more defensible signal.

The temporal graph feature set is carried forward with Random Forest for
threshold tuning and final holdout evaluation.